<a href="https://colab.research.google.com/github/Prodi-TI-IBIKesatuan/pcd2026-ti23ka-t2/blob/guntur-a2/notebooks/02_metode_klasik.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Dikerjakan oleh: Guntur (Peran A2 Metode Klasik Perbaikan Citra Malam)

# Penyelarasan Warna dan Perbaikan Citra Malam Hari — Tahap 2: metode klasik

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prodi-TI-IBIKesatuan/pcd2026-ti23ka-t2/blob/main/notebooks/02_metode_klasik.ipynb)

Kelompok ini wajib **3 metode klasik**. Pilihan yang disarankan:

- Koreksi gamma
- CLAHE pada kanal L (LAB) atau V (HSV)
- Retinex (single-scale atau multi-scale)
- White balance gray-world

Setiap metode adalah satu fungsi. Evaluasi memakai `pcd_proyek.evaluasi`, sehingga metrik sama dengan kelompok lain.

In [1]:
# Siapkan lingkungan: salin repo ini dan hubungkan Google Drive kelompok.
import pathlib
import subprocess
import sys

import cv2
import numpy as np

REPO = "Prodi-TI-IBIKesatuan/pcd2026-ti23ka-t2"
try:
    import google.colab  # noqa: F401
    DI_COLAB = True
except ImportError:
    DI_COLAB = False

if DI_COLAB:
    AKAR = pathlib.Path("/content/repo")
    if AKAR.exists():
        subprocess.run(["git", "-C", str(AKAR), "pull", "-q"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", f"https://github.com/{REPO}.git", str(AKAR)], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    FOLDER_KERJA = pathlib.Path("/content/drive/MyDrive/pcd2026-ti23ka-t2")
    if not FOLDER_KERJA.exists():
        raise FileNotFoundError(f"{FOLDER_KERJA} tidak ada. Tambahkan pintasan folder Drive kelompok ke Drive Saya (lihat PANDUAN.md).")
else:  # dijalankan di komputer sendiri dari folder notebooks/
    AKAR = pathlib.Path.cwd().resolve().parent
    FOLDER_KERJA = AKAR / "data"
sys.path.insert(0, str(AKAR))

from pcd_proyek import evaluasi as ev, metrik, util  # noqa: E402

FOLDER_CITRA = FOLDER_KERJA / "citra" if (FOLDER_KERJA / "citra").exists() else AKAR / "data" / "sample"
BERKAS_ANOTASI = FOLDER_KERJA / "anotasi.csv"
FOLDER_FIGUR = FOLDER_KERJA / "figures"
BERKAS_HASIL = FOLDER_KERJA / "hasil_metode.json"
KOLOM_ANOTASI = ['berkas_gelap', 'berkas_referensi', 'sumber']
JUMLAH_ANGGOTA = 5
print("Citra   :", FOLDER_CITRA)
print("Anotasi :", BERKAS_ANOTASI, "(ada)" if BERKAS_ANOTASI.exists() else "(belum ada)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Citra   : /content/drive/MyDrive/pcd2026-ti23ka-t2/citra
Anotasi : /content/drive/MyDrive/pcd2026-ti23ka-t2/anotasi.csv (ada)


In [2]:
anotasi = util.baca_anotasi(BERKAS_ANOTASI, KOLOM_ANOTASI)
UJI_CEPAT = None  # isi angka (mis. 20) saat mencoba-coba agar cepat; kembalikan ke None untuk hasil akhir
print(len(anotasi), "baris anotasi")

330 baris anotasi


## Metode klasik 1

**Nama metode dan alasannya:** Koreksi Gamma pada Kanal Luminance (LAB). Dipilih karena mampu meningkatkan kecerahan area bayangan gelap secara non-linear tanpa membuat area terang menjadi *overexposed*, serta menjaga informasi warna tetap terpisah pada kanal A dan B.

In [3]:
def metode_klasik_1(citra_bgr):
    """Metode klasik 1: Koreksi Gamma pada kanal L (LAB).

    Masukan: citra BGR uint8. Keluaran: citra_hasil_bgr — citra hasil perbaikan, ukuran boleh berbeda (otomatis disamakan).
    """
    citra_std = cv2.resize(citra_bgr, (512, 512), interpolation=cv2.INTER_AREA)
    lab = cv2.cvtColor(citra_std, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    gamma = 0.45
    lut = np.array([((i / 255.0) ** gamma) * 255 for i in range(256)], dtype=np.uint8)
    l_baru = cv2.LUT(l, lut)
    return cv2.cvtColor(cv2.merge([l_baru, a, b]), cv2.COLOR_LAB2BGR)

In [4]:
hasil_1 = ev.evaluasi_berreferensi(metode_klasik_1, anotasi, FOLDER_CITRA, "berkas_gelap", "berkas_referensi", batas=UJI_CEPAT)
print(hasil_1['metrik'])
NAMA_1 = "Koreksi Gamma (LAB)"
if UJI_CEPAT is None:
    ev.simpan_hasil(BERKAS_HASIL, NAMA_1, "klasik", hasil_1)

{'PSNR': 13.659171791584102, 'SSIM': 0.5699417173917583, 'jumlah': 330, 'tanpa_acuan': 0}


## Metode klasik 2

**Nama metode dan alasannya:** CLAHE (*Contrast Limited Adaptive Histogram Equalization*) pada Kanal L (LAB). Dipilih untuk meningkatkan kontras lokal dan detail tekstur pada citra malam hari tanpa memperkuat *noise* secara berlebihan berkat pembatasan *clipLimit*.

In [5]:
def metode_klasik_2(citra_bgr):
    """Metode klasik 2: CLAHE pada kanal L (LAB).

    Masukan: citra BGR uint8. Keluaran: citra_hasil_bgr — citra hasil perbaikan, ukuran boleh berbeda (otomatis disamakan).
    """
    citra_std = cv2.resize(citra_bgr, (512, 512), interpolation=cv2.INTER_AREA)
    lab = cv2.cvtColor(citra_std, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    l_baru = clahe.apply(l)
    return cv2.cvtColor(cv2.merge([l_baru, a, b]), cv2.COLOR_LAB2BGR)

In [6]:
hasil_2 = ev.evaluasi_berreferensi(metode_klasik_2, anotasi, FOLDER_CITRA, "berkas_gelap", "berkas_referensi", batas=UJI_CEPAT)
print(hasil_2['metrik'])
NAMA_2 = "CLAHE Kanal L (LAB)"
if UJI_CEPAT is None:
    ev.simpan_hasil(BERKAS_HASIL, NAMA_2, "klasik", hasil_2)

{'PSNR': 10.976016554422007, 'SSIM': 0.4198111907331537, 'jumlah': 330, 'tanpa_acuan': 0}


## Metode klasik 3

**Nama metode dan alasannya:** White Balance Gray-World + Multi-Scale Retinex (MSR). Dipilih untuk menetralkan pergeseran warna (*color cast*) akibat lampu jalan/pencahayaan malam sekaligus memulihkan pencahayaan dan detail bayangan secara multi-skala.

In [7]:
def metode_klasik_3(citra_bgr):
    """Metode klasik 3: White Balance Gray-World + Multi-Scale Retinex (MSR) teroptimasi."""
    citra_std = cv2.resize(citra_bgr, (512, 512), interpolation=cv2.INTER_AREA)
    img = citra_std.astype(np.float32) + 1.0

    # 1. White balance Gray-World
    avg_bgr = np.mean(img, axis=(0, 1))
    img_wb = img * (np.mean(avg_bgr) / avg_bgr)

    # 2. Multi-Scale Retinex cepat (blur skala besar dihitung via downscale)
    log_img = np.log10(img_wb)
    msr = np.zeros_like(img_wb)
    for sigma in (15, 80, 250):
        if sigma >= 80:
            kecil = cv2.resize(img_wb, (128, 128), interpolation=cv2.INTER_AREA)
            blur_kecil = cv2.GaussianBlur(kecil, (0, 0), sigma / 4.0)
            blur = cv2.resize(blur_kecil, (512, 512), interpolation=cv2.INTER_LINEAR)
        else:
            blur = cv2.GaussianBlur(img_wb, (0, 0), sigma)
        msr += log_img - np.log10(blur + 1.0)
    msr /= 3.0

    msr_norm = cv2.normalize(msr, None, 0, 255, cv2.NORM_MINMAX)
    return np.clip(msr_norm, 0, 255).astype(np.uint8)

In [9]:
hasil_3 = ev.evaluasi_berreferensi(metode_klasik_3, anotasi, FOLDER_CITRA, "berkas_gelap", "berkas_referensi", batas=UJI_CEPAT)
print(hasil_3['metrik'])
NAMA_3 = "Gray-World + Multi-Scale Retinex"
if UJI_CEPAT is None:
    ev.simpan_hasil(BERKAS_HASIL, NAMA_3, "klasik", hasil_3)

{'PSNR': 13.763204521977885, 'SSIM': 0.5624250568084327, 'jumlah': 330, 'tanpa_acuan': 0}


## Simpan ke GitHub

1. **Runtime → Restart session and run all**, pastikan tidak ada error.
2. **File → Save a copy in GitHub** → repo `Prodi-TI-IBIKesatuan/pcd2026-ti23ka-t2` → **branch Anda** (bukan `main`) → path tetap `notebooks/02_metode_klasik.ipynb`.
3. Buka *pull request* ke `main` di github.com. Langkah lengkap: `PANDUAN.md`.